# Experiment 1: Pre-Normalised Response Analysis
## Validating Divisive Normalization Theory with Gaussian Process Tuning

### Intellectual Foundation

This experiment validates three theoretical predictions about what happens **before** divisive normalization (DN) is applied to a population of neurons whose tuning functions are drawn from Gaussian processes (GPs) with location-dependent lengthscales.

The full model works as follows:
1. Each neuron n has tuning functions f_{n,k}(θ) at each spatial location k, drawn from GPs
2. The **pre-normalised response** to a set of stimuli S at orientations θ is: r^pre_n(S,θ) = exp(Σ_{k∈S} f_{n,k}(θ_k))
3. **Divisive normalization** then computes: r_n = σ² + r^pre_n / D(S,θ), where D(S,θ) = σ² + N⁻¹ Σ_j r^pre_j

The three claims tested:

**Part A — Exponential Growth (§3.5, Eq. 11):** The expected pre-normalised response grows exponentially with set size l: E[r^pre] = ḡ^l, where ḡ = exp(σ²_f/2). This follows from Jensen's inequality applied to the log-normal distribution of r^pre.

**Part B — Denominator Concentration (§5.4):** The normalisation denominator D(S,θ) concentrates around ḡ^l as N→∞, with coefficient of variation (CV) proportional to 1/√N. This is why DN effectively makes the denominator stimulus-independent — the foundation for efficient decoding.

**Part C — Mixed Selectivity (§3.3, Def. 4.3):** Location-dependent GP lengthscales break the separability of each neuron's response into orientation × location factors. The SVD-based separability index S = σ₁²/Σσ²_i measures this: S < 0.8 indicates genuinely mixed-selective neurons.

### Why This Matters
The exponential growth of r^pre is what makes DN necessary — without normalisation, responses would explode with set size. The denominator concentration is what makes DN *useful* — it means the denominator is approximately a known function of set size alone, enabling tractable decoding. And mixed selectivity is the signature of a rich, high-dimensional neural code that supports flexible readout.


In [ ]:
# ============================================================
# SHARED DEPENDENCIES & CONFIGURATION
# ============================================================
# All experiments below import from this cell.
# Modify parameters here to explore the model.

import numpy as np
from itertools import combinations
from scipy import stats as sp_stats
import matplotlib.pyplot as plt
import time

np.random.seed(42)
plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.size': 11,
    'axes.labelsize': 12, 'axes.titlesize': 13,
    'figure.dpi': 120,
})

# --- CONFIGURATION (MODIFY THESE) ---
N_NEURONS = 100              # Number of neurons in the population
N_ORIENTATIONS = 200         # Resolution of orientation sampling
N_LOCATIONS = 8              # Number of spatial locations (L)
SET_SIZES = [2, 4, 6, 8]    # Memory load conditions to test
SEED = 42                    # Random seed for reproducibility

# Gaussian Process parameters for tuning functions
LAMBDA_BASE = 0.3            # Base lengthscale of GP (controls tuning width)
SIGMA_LAMBDA = 0.5           # Variability in lengthscale across locations
GAIN_VARIABILITY = 0.2       # Variability in neuron gain

# Derived theory constants
SIGMA_F_SQUARED = 1.0        # Variance of GP (σ²_f) — set to 1 by convention
G_BAR_THEORY = np.exp(SIGMA_F_SQUARED / 2)  # ≈ 1.6487

# Part B: sampling parameters
N_UNIQUE_TARGET = 250        # Target number of unique (S,θ) samples per set size

# Part C: separability threshold
SEPARABILITY_THRESHOLD = 0.8  # S < this → "mixed selective"


---

## Core Model: Gaussian Process Neuron Population

### What is occurring
We generate a population of N neurons. Each neuron n has L tuning functions f_{n,k}(θ), one per spatial location k, sampled from Gaussian processes. The GP kernel has a **location-dependent lengthscale** — this is the key modelling choice that produces mixed selectivity.

The lengthscale λ_k for location k is drawn from: log(λ_k) ~ N(log(λ_base), σ²_λ). A short lengthscale means sharp tuning (the neuron is very selective for orientation at that location); a long lengthscale means broad tuning.

### Why we build this
This is the generative model underlying all three experiment parts. The GP framework is powerful because:
1. It gives us smooth, realistic tuning functions without hand-crafting them
2. Location-dependent lengthscales naturally produce the mixed selectivity that we test in Part C
3. The log-normal structure of r^pre = exp(Σ f_{n,k}) is what gives rise to the exponential growth tested in Part A

### What to play with
- `LAMBDA_BASE`: Smaller → sharper tuning → more orientation-selective neurons
- `SIGMA_LAMBDA`: Larger → more variability across locations → more mixed selectivity
- `N_NEURONS`: Larger → better denominator concentration (Part B)


In [ ]:
# ============================================================
# CORE MODEL: Generate Neuron Population from Gaussian Processes
# ============================================================

def gp_kernel_squared_exponential(theta_grid, lengthscale):
    """Squared exponential (RBF) kernel on the circle."""
    n = len(theta_grid)
    # Circular distance
    diff = theta_grid[:, None] - theta_grid[None, :]
    circ_dist = np.abs(np.angle(np.exp(1j * diff)))
    return np.exp(-0.5 * (circ_dist / lengthscale)**2)

def generate_neuron_population(n_neurons, n_orientations, n_locations, 
                                base_lengthscale, lengthscale_variability,
                                seed, gain_variability=0.2):
    """
    Generate a population of neurons with GP-sampled tuning functions.
    
    Each neuron has L tuning functions f_{n,k}(θ) drawn from GPs with
    location-dependent lengthscales.
    
    Returns list of dicts, each with:
        'f_samples': (L, n_orientations) array of tuning function values
        'lengthscales': (L,) array of per-location lengthscales
    """
    rng = np.random.default_rng(seed)
    theta_grid = np.linspace(-np.pi, np.pi, n_orientations, endpoint=False)
    
    population = []
    for n in range(n_neurons):
        # Location-dependent lengthscales (log-normal)
        log_lambdas = np.log(base_lengthscale) + lengthscale_variability * rng.standard_normal(n_locations)
        lambdas = np.exp(log_lambdas)
        
        # Per-neuron gain variability
        gain = np.exp(gain_variability * rng.standard_normal())
        
        f_samples = np.zeros((n_locations, n_orientations))
        for k in range(n_locations):
            # GP kernel at this location
            K = gp_kernel_squared_exponential(theta_grid, lambdas[k])
            # Add small jitter for numerical stability
            K += 1e-6 * np.eye(n_orientations)
            # Sample from GP
            L_chol = np.linalg.cholesky(K)
            f_samples[k] = gain * (L_chol @ rng.standard_normal(n_orientations))
        
        population.append({
            'f_samples': f_samples,
            'lengthscales': lambdas,
            'gain': gain,
        })
    
    return population

# Generate the population
t0 = time.time()
population = generate_neuron_population(
    N_NEURONS, N_ORIENTATIONS, N_LOCATIONS,
    LAMBDA_BASE, SIGMA_LAMBDA, SEED, GAIN_VARIABILITY
)
F_stacked = np.stack([neuron['f_samples'] for neuron in population])  # (N, L, n_θ)
print(f"Generated {N_NEURONS} neurons in {time.time()-t0:.2f}s")
print(f"F_stacked shape: {F_stacked.shape}  (N_neurons × L_locations × n_orientations)")
print(f"Theory: ḡ = exp(σ²_f/2) = {G_BAR_THEORY:.4f}")


In [ ]:
# Visualise example tuning functions

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
theta_grid = np.linspace(-180, 180, N_ORIENTATIONS, endpoint=False)

for row, neuron_idx in enumerate([0, 1]):
    neuron = population[neuron_idx]
    for col, loc_k in enumerate(range(4)):
        ax = axes[row, col]
        ax.plot(theta_grid, neuron['f_samples'][loc_k], linewidth=1.5)
        ax.set_title(f"Neuron {neuron_idx}, Loc {loc_k}\nλ={neuron['lengthscales'][loc_k]:.3f}")
        ax.set_xlabel('θ (°)') if row == 1 else None
        ax.set_ylabel('f(θ)') if col == 0 else None
        ax.grid(True, alpha=0.3)

plt.suptitle('GP-Sampled Tuning Functions (different lengthscales per location)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("KEY: Different lengthscales at different locations → the neuron's orientation tuning")
print("     varies across space → this is what breaks separability (Part C)")


---

## Shared Helper Functions

These utility functions are used across all three experiment parts.


In [ ]:
# ============================================================
# SHARED HELPERS
# ============================================================

def compute_pre_normalized_response(f_samples, subset):
    """
    Compute r^pre = exp(Σ_{k∈S} f_{n,k}(θ_k)) for all orientations at once.
    
    For simplicity, we use the SAME orientation index for all locations
    (equivalent to a single orientation vector). The mean over orientations
    gives E_θ[r^pre].
    
    Args:
        f_samples: (L, n_θ) tuning functions for one neuron
        subset: tuple of active location indices
    Returns:
        (n_θ,) array of r^pre values (one per orientation)
    """
    # Sum f across active locations
    f_sum = np.sum(f_samples[list(subset), :], axis=0)  # (n_θ,)
    return np.exp(f_sum)

def compute_r_pre_at_config(F_stacked, subset, active_theta_indices):
    """
    Compute r^pre for ALL neurons at a specific (S, θ) configuration.
    
    Args:
        F_stacked: (N, L, n_θ) all neurons' tuning functions
        subset: tuple of active location indices  
        active_theta_indices: list of orientation indices, one per active location
    Returns:
        (N,) array of r^pre values (one per neuron)
    """
    N = F_stacked.shape[0]
    f_sum = np.zeros(N)
    for i, k in enumerate(subset):
        f_sum += F_stacked[:, k, active_theta_indices[i]]
    return np.exp(f_sum)

def exp_fit(set_sizes, values):
    """
    Fit log(values) = intercept + slope · l.
    Returns g_bar = exp(slope), R², and prediction function.
    
    This fits the Jensen's inequality prediction E[r^pre] = ḡ^l.
    """
    log_v = np.log(np.maximum(values, 1e-30))
    slope, intercept, r_value, _, _ = sp_stats.linregress(set_sizes, log_v)
    return {
        'g_bar': np.exp(slope),
        'intercept': intercept,
        'slope': slope,
        'r_squared': r_value ** 2,
        'predict': lambda l: np.exp(intercept + slope * np.asarray(l, dtype=float)),
    }

def band_stats(values):
    """Summary statistics for a 1-D array."""
    q05, q25, q75, q95 = np.percentile(values, [5, 25, 75, 95])
    mu = np.mean(values)
    return {
        'mean': mu, 'median': np.median(values), 'std': np.std(values),
        'cv': np.std(values) / (mu + 1e-30),
        'q05': q05, 'q25': q25, 'q75': q75, 'q95': q95,
    }

print("Helpers loaded.")


---

## Part A — Exponential Growth of E[r^pre] (§3.5, Eq. 11)

### What is occurring
For each neuron, we exhaustively enumerate ALL C(L, l) location subsets at each set size l. At each subset, we compute r^pre = exp(Σ_{k∈S} f_{n,k}(θ_k)) and average over orientations and subsets. Then we average across neurons to get the population mean.

### The theoretical prediction
Since each f_{n,k}(θ_k) is drawn from a GP with variance σ²_f, the sum Σ_{k∈S} f_{n,k} has variance approximately l·σ²_f (assuming approximate independence across locations). Therefore r^pre = exp(Σ f) is log-normal, and by Jensen's inequality:

**E[r^pre] = exp(l · σ²_f / 2) = ḡ^l**

where ḡ = exp(σ²_f / 2) ≈ 1.65 for σ²_f = 1.

This exponential growth is **the reason divisive normalization is necessary**: without it, neural responses would blow up as more items are stored.

### Why we do this experiment
We validate that the theoretical ḡ matches the empirical fit. A high R² confirms the exponential law holds. Any deviation from ḡ ≈ 1.65 would indicate that the independence assumption between locations is violated or that gain variability introduces corrections.

### What to play with
- `SIGMA_F_SQUARED` (indirectly via `GAIN_VARIABILITY`): Changes the theoretical ḡ
- `SET_SIZES`: Try [1, 2, 3, 4, 5, 6, 7, 8] for finer resolution
- `N_LOCATIONS`: With fewer locations, there are fewer subsets to enumerate (faster, but less averaging)


In [ ]:
# ============================================================
# PART A: EXPONENTIAL GROWTH — Exhaustive Enumeration
# ============================================================

print("Part A: Exhaustive enumeration of pre-normalised responses...")
t0 = time.time()

neuron_means = {l: [] for l in SET_SIZES}

for n_idx, neuron in enumerate(population):
    f = neuron['f_samples']  # (L, n_θ)
    for l in SET_SIZES:
        subset_avgs = []
        for subset in combinations(range(N_LOCATIONS), l):
            R_pre = compute_pre_normalized_response(f, subset)
            subset_avgs.append(np.mean(R_pre))  # Average over orientations
        neuron_means[l].append(np.mean(subset_avgs))  # Average over subsets

# Population statistics
pop_means = {l: np.mean(neuron_means[l]) for l in SET_SIZES}
pop_stds  = {l: np.std(neuron_means[l])  for l in SET_SIZES}

# Exponential fit to population MEANS
fit_a = exp_fit(SET_SIZES, [pop_means[l] for l in SET_SIZES])

print(f"  Completed in {time.time()-t0:.1f}s")
print(f"  ḡ (fitted) = {fit_a['g_bar']:.4f}  (theory: {G_BAR_THEORY:.4f})")
print(f"  R² = {fit_a['r_squared']:.6f}")
for l in SET_SIZES:
    print(f"    l={l}: E[r^pre] = {pop_means[l]:.4f} ± {pop_stds[l]:.4f}")


In [ ]:
# ============================================================
# PART A: PLOT — Exponential Growth
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_yscale('log')

means = [pop_means[l] for l in SET_SIZES]
stds  = [pop_stds[l]  for l in SET_SIZES]

ax.errorbar(SET_SIZES, means, yerr=stds, fmt='o-', color='#E74C3C',
            lw=2, ms=10, capsize=5, label='Population mean ± SD', zorder=5)

# Fit line
l_fine = np.linspace(min(SET_SIZES) - 0.5, max(SET_SIZES) + 0.5, 100)
ax.plot(l_fine, fit_a['predict'](l_fine), '--', color='gray', lw=2,
        label=f'Fit: {np.exp(fit_a["intercept"]):.2f} × {fit_a["g_bar"]:.3f}$^l$'
              f'  (R²={fit_a["r_squared"]:.4f})')

# Theory prediction
theory_pred = [G_BAR_THEORY**l for l in SET_SIZES]
ax.plot(SET_SIZES, theory_pred, 's', color='#27AE60', ms=10, mew=2, mfc='none',
        label=f'Theory: ḡ$^l$, ḡ={G_BAR_THEORY:.3f}', zorder=4)

ax.set_xlabel('Set size $l$', fontsize=13)
ax.set_ylabel(r'$\mathbb{E}[r^{\mathrm{pre}}]$  (log scale)', fontsize=13)
ax.set_title(f'Part A: Exponential Growth of Pre-DN Response  (N={N_NEURONS})', fontsize=14)
ax.set_xticks(SET_SIZES)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Theory annotation
ax.text(0.02, 0.98,
        f"Jensen's inequality (§3.5):\n"
        f"$\mathbb{{E}}[r^{{\mathrm{{pre}}}}] = \bar{{g}}^l$\n"
        f"$\bar{{g}} = e^{{\sigma^2_f/2}} \approx {G_BAR_THEORY:.3f}$\n"
        f"Observed: $\bar{{g}} = {fit_a['g_bar']:.3f}$",
        transform=ax.transAxes, fontsize=10, va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', fc='#D5F5E3', ec='#27AE60', alpha=0.9))

plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print(f"  The fitted ḡ = {fit_a['g_bar']:.4f} vs theory {G_BAR_THEORY:.4f}")
print(f"  Exponential growth CONFIRMED with R² = {fit_a['r_squared']:.6f}")
print(f"  → This is WHY divisive normalisation is needed: without it, responses explode with load.")


---

## Part B — Denominator Concentration (§5.4)

### What is occurring
We sample random stimulus configurations (S, θ) — where S is a random subset of locations and θ is a random orientation vector — and compute the normalisation denominator:

D(S, θ) = N⁻¹ Σ_j r^pre_j(S, θ)

This is the population-average pre-normalised response, which becomes the denominator of the DN equation.

### The theoretical prediction
By the law of large numbers, as N→∞, the average over neurons concentrates around the expected value:

D(S, θ) → E_n[r^pre_n(S, θ)] ≈ ḡ^l

with fluctuations scaling as CV ∝ 1/√N. This means the denominator becomes **effectively deterministic** — it depends on set size l but NOT on the specific stimulus configuration (S, θ).

### Why we do this experiment
This is the theoretical foundation for efficient decoding. If D were highly variable across stimuli, the brain would need to estimate it on every trial — an impossible task. But because D concentrates, decoding can treat the denominator as a known constant (given l), vastly simplifying the inference problem.

### What to play with
- `N_NEURONS`: Larger → lower CV (better concentration) — try 50 vs 200 vs 500
- `N_UNIQUE_TARGET`: More samples → more precise CV estimates
- Compare D's ḡ with Part A's ḡ — they should match (same underlying distribution)


In [ ]:
# ============================================================
# PART B: DENOMINATOR CONCENTRATION — Random (S,θ) Sampling
# ============================================================

from math import comb

print("Part B: Random (S, θ) sampling of normalisation denominator...")
t0 = time.time()

rng_b = np.random.default_rng(SEED + 2000)

D_per_l = {}       # D(S,θ) values at each set size
neuron_per_l = {}   # Per-neuron averages

for l in SET_SIZES:
    n_possible = comb(N_LOCATIONS, l)
    n_sub = min(n_possible, N_UNIQUE_TARGET)
    n_theta = max(1, int(np.ceil(N_UNIQUE_TARGET / n_sub)))
    
    # Generate subsets
    if n_possible <= N_UNIQUE_TARGET:
        subsets = list(combinations(range(N_LOCATIONS), l))
    else:
        subsets = [tuple(sorted(rng_b.choice(N_LOCATIONS, size=l, replace=False)))
                   for _ in range(n_sub)]
    
    # Sample responses
    responses = []  # Will be (n_samples, N)
    for _ in range(n_theta):
        theta_idx_full = rng_b.integers(0, N_ORIENTATIONS, size=N_LOCATIONS)
        for subset in subsets:
            active_theta = [theta_idx_full[k] for k in subset]
            responses.append(compute_r_pre_at_config(F_stacked, subset, active_theta))
    
    responses = np.stack(responses)              # (n_samples, N)
    D_per_l[l] = np.mean(responses, axis=1)      # D at each stimulus config
    neuron_per_l[l] = np.mean(responses, axis=0)  # Per-neuron average

# Statistics and fit
D_stats = {l: band_stats(D_per_l[l]) for l in SET_SIZES}
D_means = {l: D_stats[l]['mean'] for l in SET_SIZES}
D_fit = exp_fit(SET_SIZES, [D_means[l] for l in SET_SIZES])

print(f"  Completed in {time.time()-t0:.1f}s")
print(f"  D fit: ḡ = {D_fit['g_bar']:.4f}  R² = {D_fit['r_squared']:.6f}")
for l in SET_SIZES:
    cv = D_stats[l]['cv']
    print(f"    l={l}: D mean = {D_stats[l]['mean']:.4f}  CV = {cv:.4f}")


In [ ]:
# ============================================================
# PART B: PLOT — Denominator Concentration
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: D(S,θ) dot-band plot ---
ax1.set_yscale('log')
rng_plot = np.random.default_rng(0)
colors = plt.cm.Set2(np.linspace(0, 1, len(SET_SIZES)))

for i, l in enumerate(SET_SIZES):
    vals = D_per_l[l]
    x_jitter = l + rng_plot.uniform(-0.2, 0.2, len(vals))
    ax1.scatter(x_jitter, vals, s=12, alpha=0.35, color=colors[i], edgecolors='none')
    
    stats = D_stats[l]
    ax1.plot([l - 0.3, l + 0.3], [stats['median']] * 2, 'k-', lw=2.5)
    ax1.plot([l, l], [stats['q25'], stats['q75']], 'k-', lw=1.5)

# Fit line
l_fine = np.linspace(min(SET_SIZES) - 0.5, max(SET_SIZES) + 0.5, 100)
ax1.plot(l_fine, D_fit['predict'](l_fine), '--', color='gray', lw=2,
         label=f'Fit: ḡ={D_fit["g_bar"]:.3f}  (R²={D_fit["r_squared"]:.4f})')

ax1.set_xlabel('Set size $l$', fontsize=12)
ax1.set_ylabel(r'$D(S, 	heta)$  (log scale)', fontsize=12)
ax1.set_title('Denominator per stimulus configuration', fontsize=13)
ax1.set_xticks(SET_SIZES)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Right: CV bar chart ---
cvs = [D_stats[l]['cv'] for l in SET_SIZES]
ax2.bar(SET_SIZES, cvs, width=0.6, color='#3498DB', alpha=0.7, edgecolor='black')
for l, cv in zip(SET_SIZES, cvs):
    ax2.text(l, cv + 0.003, f'{cv:.3f}', ha='center', fontsize=10, fontweight='bold')

ax2.set_xlabel('Set size $l$', fontsize=12)
ax2.set_ylabel(r'CV of $D(S, 	heta)$', fontsize=12)
ax2.set_title('Denominator concentration quality', fontsize=13)
ax2.grid(True, alpha=0.3, axis='y')

ax2.text(0.02, 0.98,
         f"§5.4: As N → ∞,\n"
         f"D → ḡ$^l$ (deterministic)\n"
         f"CV ∝ 1/√N\n\n"
         f"N = {N_NEURONS}: CV range\n"
         f"[{min(cvs):.3f}, {max(cvs):.3f}]",
         transform=ax2.transAxes, fontsize=10, va='top', ha='left',
         bbox=dict(boxstyle='round,pad=0.4', fc='#EBF5FB', ec='#3498DB', alpha=0.9))

plt.suptitle(f'Part B: Denominator Concentration  (N={N_NEURONS})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print(f"  D's ḡ = {D_fit['g_bar']:.4f} matches Part A's ḡ = {fit_a['g_bar']:.4f} — same exponential growth")
print(f"  CV is {min(cvs):.3f}–{max(cvs):.3f} — denominator is fairly concentrated even at N={N_NEURONS}")
print(f"  → Increasing N would reduce CV further (∝ 1/√N)")
print(f"  → This is WHY the decoder can treat D as approximately known given l")


---

## Part C — Mixed Selectivity (§3.3, Def. 4.3)

### What is occurring
For each neuron, we form its response matrix R ∈ ℝ^{n_θ × L}, where each column is the tuning function at one location. We compute the SVD and measure the **separability index**:

S = σ₁² / Σ_i σ²_i

where σ_i are the singular values.

- S = 1 → perfectly separable (rank-1): the neuron's tuning is a product of an orientation profile × a location profile
- S < 0.8 → mixed selective: orientation tuning changes qualitatively across locations

### The theoretical prediction
Because GP lengthscales vary across locations, the tuning function shape (not just amplitude) changes from one location to another. This guarantees that most neurons will have S ≪ 1 — they are **conjunctive** for orientation × location.

### Why we do this experiment
Mixed selectivity is computationally powerful: it supports linear readout of arbitrary combinations of stimulus features. If neurons were separable (S ≈ 1), the code would be low-dimensional and inflexible. The GP model with variable lengthscales naturally produces the rich, high-dimensional code needed for flexible working memory.

### What to play with
- `SIGMA_LAMBDA = 0`: Forces all locations to have the same lengthscale → separability approaches 1
- `SIGMA_LAMBDA = 1.0`: Extreme variability → S pushed toward 0
- `SEPARABILITY_THRESHOLD`: Where you draw the line between "separable" and "mixed"


In [ ]:
# ============================================================
# PART C: MIXED SELECTIVITY — SVD Separability Analysis
# ============================================================

print("Part C: SVD-based separability analysis...")
t0 = time.time()

all_separabilities = []

for neuron in population:
    f = neuron['f_samples']  # (L, n_θ)
    # Form response matrix: each column is a location's tuning curve
    # We exponentiate to get the response (not log-response)
    R_matrix = np.exp(f).T  # (n_θ, L) — rows = orientations, cols = locations
    
    # Subtract mean (optional: centers the matrix)
    R_centered = R_matrix - R_matrix.mean(axis=0, keepdims=True)
    
    # SVD
    U, singular_values, Vt = np.linalg.svd(R_centered, full_matrices=False)
    
    # Separability index
    sv_squared = singular_values**2
    S_index = sv_squared[0] / np.sum(sv_squared) if np.sum(sv_squared) > 0 else 1.0
    all_separabilities.append(S_index)

all_separabilities = np.array(all_separabilities)

# Classification
n_mixed = np.sum(all_separabilities < SEPARABILITY_THRESHOLD)
percent_mixed = 100 * n_mixed / N_NEURONS

print(f"  Completed in {time.time()-t0:.2f}s")
print(f"  Mean separability S = {np.mean(all_separabilities):.4f}")
print(f"  Median S = {np.median(all_separabilities):.4f}")
print(f"  Mixed selective (S < {SEPARABILITY_THRESHOLD}): {percent_mixed:.1f}% ({n_mixed}/{N_NEURONS})")


In [ ]:
# ============================================================
# PART C: PLOT — Separability Histogram
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

ax.hist(all_separabilities, bins=25, color='#9B59B6', alpha=0.7, edgecolor='white', linewidth=0.8)
ax.axvline(SEPARABILITY_THRESHOLD, color='red', ls='--', lw=2.5, 
           label=f'S = {SEPARABILITY_THRESHOLD} threshold')
ax.axvline(np.mean(all_separabilities), color='#2E86AB', ls='-', lw=2.5,
           label=f'Mean: {np.mean(all_separabilities):.3f}')

ax.set_xlabel('Separability index $S$', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title(f'Part C: Mixed Selectivity  ({percent_mixed:.0f}% mixed, N={N_NEURONS})', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

ax.text(0.02, 0.98,
        f"Def. 4.3: S = σ₁²/Σσ²ᵢ\n"
        f"S = 1 → separable\n"
        f"S < {SEPARABILITY_THRESHOLD} → mixed selective\n\n"
        f"σ_λ = {SIGMA_LAMBDA} (lengthscale var.)\n"
        f"Mean S = {np.mean(all_separabilities):.3f}",
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round,pad=0.4', fc='#F5EEF8', ec='#9B59B6', alpha=0.9))

plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print(f"  {percent_mixed:.0f}% of neurons are mixed selective (S < {SEPARABILITY_THRESHOLD})")
print(f"  This means orientation tuning CHANGES SHAPE across locations")
print(f"  → Location-dependent lengthscales break separability, as predicted")
print(f"  → This produces the high-dimensional code needed for flexible readout")


---

## Summary of Validated Claims

Run the experiments above and fill in the observed values:

| Claim | Prediction | What to check |
|-------|-----------|---------------|
| **A: Exponential growth** | E[r^pre] = ḡ^l, ḡ ≈ 1.65 | Compare fitted ḡ and R² from Part A |
| **B: Denominator concentration** | D → ḡ^l, CV ∝ 1/√N | Check CV range from Part B |
| **C: Mixed selectivity** | S < 0.8 for most neurons | Check % mixed from Part C |

### Causal chain
1. GP tuning functions with variable lengthscales → **mixed selectivity** (Part C)
2. Exponential of summed GPs → **log-normal r^pre** → **exponential growth** with set size (Part A)
3. Averaging over neurons → **law of large numbers** → **denominator concentrates** (Part B)
4. Concentrated denominator → DN is tractable → **efficient decoding** is possible
